# 03 — Vehicle Wholesale/Fleet Cleaning

This notebook parses the report-style `Vehicle_Wholesale_Data` sheet into an in-memory monthly long table. Wholesale and Fleet are kept separate, and subtotal/total rows are explicitly marked.

It does **not** merge with PIO sales, calculate penetration rate, build forecasts, or export processed data.

> **Data safety:** Clear all outputs before committing. The committed notebook must contain code and markdown only.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.load_excel import detect_vehicle_report_structure
from src.vehicle_wholesale_cleaning import (
    build_year_month_column_map,
    clean_vehicle_wholesale,
    load_vehicle_wholesale,
    split_model_and_total_rows,
    vehicle_cleaning_summaries,
)

## 1. Load the raw report and detect sections

The sheet is read with `header=None`. Detected row numbers are Excel-style and remain estimates until visually confirmed.

In [ ]:
workbook_path, vehicle_raw = load_vehicle_wholesale(
    PROJECT_ROOT / "data" / "raw"
)
report_structure = detect_vehicle_report_structure(vehicle_raw)

layout_summary = pd.DataFrame([
    {
        "channel": section["section"],
        "start_row": section["start_row"],
        "end_row": section["end_row"],
        "brand_block_rows": [
            item["row_number"] for item in section["brand_blocks"]
        ],
        "subtotal_rows": [
            item["row_number"] for item in section["subtotal_rows"]
        ],
        "year_header_rows": [
            item["row_number"] for item in section["year_header_rows"]
        ],
        "month_header_rows": [
            item["row_number"] for item in section["month_header_rows"]
        ],
    }
    for section in report_structure["sections"]
])

print(f"Workbook: {workbook_path.name}")
print(f"Raw vehicle sheet shape: {vehicle_raw.shape}")
display(layout_summary)

## 2. Reconstruct year-month columns

Annual `Total` columns are intentionally excluded. When a section has a blank or merged year label, the year is inferred only from the same physical Excel column in another detected section.

In [ ]:
year_month_map = build_year_month_column_map(
    vehicle_raw, report_structure
)
display(year_month_map)

display(Markdown("### Reconstructed month coverage"))
display(
    year_month_map.groupby("channel", as_index=False)
    .agg(
        first_month=("month", "min"),
        last_month=("month", "max"),
        month_count=("month", "nunique"),
        inferred_year_columns=("year_was_inferred", "sum"),
    )
)

## 3. Create the in-memory long table

Primary fields are `month`, `channel`, `brand_group`, `model`, `model_code`, `vehicle_volume`, `source_row`, and `is_total_row`. Extra source/status fields preserve auditability.

In [ ]:
vehicle_long = clean_vehicle_wholesale(
    vehicle_raw, report_structure
)
vehicle_models, vehicle_totals = split_model_and_total_rows(vehicle_long)

print(f"Long-table shape: {vehicle_long.shape}")
print(f"Model-level rows: {len(vehicle_models):,}")
print(f"Subtotal/total rows: {len(vehicle_totals):,}")
display(vehicle_long.head(20))

display(Markdown("### Long-table data types"))
display(
    vehicle_long.dtypes.astype(str)
    .rename_axis("column")
    .reset_index(name="dtype")
)

### Model rows and total rows remain separate

Model-level analysis should use `vehicle_models`. The total table is retained for reconciliation and reporting checks, not mixed into model-level sums.

In [ ]:
display(Markdown("**Model-level sample**"))
display(vehicle_models.head(10))

display(Markdown("**Subtotal/total labels**"))
display(
    vehicle_totals[
        ["channel", "brand_group", "total_label", "source_row"]
    ].drop_duplicates().sort_values(["channel", "source_row"])
)

## 4. Cleaning and completeness checks

Monthly channel totals below sum model rows only, preventing subtotal double counting. Blank source cells remain missing rather than being assumed to be zero.

In [ ]:
summaries = vehicle_cleaning_summaries(vehicle_long)

for title, key in [
    ("Row counts by channel and brand group", "row_counts_by_channel_brand"),
    ("Monthly model-level vehicle volume by channel", "monthly_vehicle_volume_by_channel"),
    ("Total rows versus model rows", "total_rows_vs_model_rows"),
    ("Missing or nonnumeric volume values", "volume_quality"),
    ("Months covered", "months_covered"),
]:
    display(Markdown(f"### {title}"))
    display(summaries[key])

## 5. Validation assertions

These checks validate structure and types without treating blank future/report cells as errors.

In [ ]:
assert set(vehicle_long["channel"].dropna()) == {"Wholesale", "Fleet"}
assert not vehicle_models.empty
assert not vehicle_totals.empty
assert vehicle_totals["is_total_row"].all()
assert vehicle_models["is_total_row"].eq(False).all()
assert vehicle_long["month"].notna().all()
assert vehicle_long["month"].dt.day.eq(1).all()
assert vehicle_long.loc[
    vehicle_long["volume_status"] == "numeric", "vehicle_volume"
].notna().all()
assert not vehicle_long["volume_status"].eq("nonnumeric").any()
assert year_month_map.groupby("channel")["month"].nunique().eq(24).all()

print("All in-memory vehicle cleaning checks passed.")

## 6. Assumptions and uncertainties

### Working parsing assumptions

- Wholesale and Fleet are separate channels and are parsed independently.
- The detected `Model` and `Model Code` header cells identify the model columns; no manual model mapping is applied.
- HMA, GMA, and KUS are retained as report-level `brand_group` labels. Their relationship to PIO company/brand codes is not assumed.
- PIO `PIS_SERI` appears to align closely with this report's model code. It is useful for validating mappings and variant differences, but matching should still combine brand, model name, month, and model-code evidence.
- HMA Total, GMA Total, H+G Total, KUS Total, and Grand Total are marked with `is_total_row=True` and excluded from model-level summaries.
- Annual `Total` columns are excluded because this table is monthly long format.
- Blank monthly cells remain missing. They are not converted to zero because blank may mean no volume, unavailable data, or a future/unreported month.
- `month` uses the first day of each reconstructed month as a standard monthly timestamp.
- `source_row` is the one-based Excel row number and `source_column` is the Excel column label for auditability.

### Uncertain decisions requiring review

- The Fleet 2025 year header is blank in the raw layout. It is inferred from the matching Wholesale columns and repeated Jan–Dec positions. This should be confirmed visually and with Mobis.
- Model code is not assumed unique: gasoline, HEV, PHEV, and EV variants may share codes.
- Confirm whether blank monthly cells should remain missing or represent zero vehicles.
- Confirm whether Wholesale and Fleet are mutually exclusive and whether either should be combined later.
- Confirm whether report subtotals are authoritative reconciliation values and whether adjustments exist outside model rows.
- Confirm the exact business meaning of HMA, GMA, H+G, and KUS before later matching to PIO sales.

### Future use—not performed here

After review, model-level vehicle volume may be matched to PIO quantity and used to evaluate a relationship such as `PIO quantity ≈ vehicle volume × penetration rate`. Penetration rate must not be calculated until the model/brand match and denominator are valid.

**Stopping point:** no PIO merge, penetration calculation, forecasting model, dashboard, CSV, or processed dataset is created.

> **Before committing:** clear every output and execution count.